In [ ]:
# allows update of external libraries without need to reload package
%load_ext autoreload
%autoreload 2

# Terrain complexity assessment and transferability

## Summary

Demonstrate and visualize the application of T-RIX assessment of the transferability of wind assessments.

## Results

### Key results

- Examples are sunny-weather examples similar to the RIX demonstration.
- Critical parts:
  - Reprojection from EPSG:4326 and subsequent interpolation may introduce artifacts.

### Details

1. Requirements:
   - A map given as a `xarray.DataArray` given in its own coordinate system with coordinates `("y", "x")`.
   - If the map is given in a particular `CRS`, then keep this one, too.
   - A location on the map encoded in a `LocationCCS` object (given as easting and northing). `zone` is for the computations.
1. Parameters:
   - `R_km`: Radius [km] of the disc of evaluation. TR6: 3.5km.
   - `dr_km`: Stepsize between points on each ray. `R_km` shall be a multiple of `dr_km`.
1. Free parameters:
   - `slope_critical`: Threshold [m/m] at which a slope is considered steep. This is no angle. TR6: 0.033.

## Remarks

1. Plots:
   - `plot_profile_and_steep_segments`: The slopes and its mask of steep slopes represent left-labelled intervals, and are plotted for simplicity at their label, i.e. at the left bound. The artifact is of visual nature.

## Imports and Setup

In [ ]:
import ergaleiothiki
import matplotlib.pyplot as plt
import numpy as np

# import rioxarray
import rasterio

import phoibe.geography.crs
import phoibe.synthetic_data.fields
import phoibe.synthetic_data.sites
from phoibe.geography.complexity.rix import (
    NaNPolicy,
    RayGeometry,
    RayProfile,
    RayResult,
    RegularGridXYSampler,
    analyse,
    analyzer,
)

## Create scenario

## Create map and equip w/ CRS

In [ ]:
NX, NY = 201, 201
DX, DY = 10, 10
FREQ_X, FREQ_Y = 0.002, 0.002
BOUNDS = (11, 51.0, 14, 53.0)
CRS_TO = "32633"

LOCATION = ergaleiothiki.perdix.LocationCCS(easting=0, northing=0, zone=32)
R_km = 2
DR_km = 0.01
THETA = 90

eggbox = phoibe.synthetic_data.fields.make_eggbox_field(nx=NX, ny=NY, dx=DX, dy=DY, freq_x=FREQ_X, freq_y=FREQ_Y) * 500
eggbox = phoibe.synthetic_data.fields.make_field_rio(da=eggbox, bounds=BOUNDS, crs="EPSG:4326", dtype="float")
eggbox, meta_reproject = phoibe.geography.crs.reproject.reproject_rasterdata(
    da=eggbox, crs_to=CRS_TO, resolution=500, resampling=rasterio.warp.Resampling.nearest
)

meta_reproject

### Create sites

In [ ]:
sites = phoibe.synthetic_data.sites.make_sites(
    sites=7, bounds=eggbox.rio.bounds(), buffer=3e4, seed=23, crs=eggbox.rio.crs
)

figure, axes = phoibe.geography.plot.plot_raster(da=eggbox)
sites.plot(color="r", ax=axes)
for _k, row in sites.iterrows():
    axes.annotate(xy=(row.geometry.x, row.geometry.y), text=row["name"], fontsize=17)
sites

## Analyse

In [ ]:
analyzer._DEFAULTS

In [ ]:
config = analyzer._DEFAULTS.copy()
config.update({"crs": eggbox.rio.crs.to_authority(), "slope_critical": 0.01, "R_km": 17.0, "n_angles": 32})
rix_analyzer = analyzer.RIXAnalyzer(config=config)

rix_results = rix_analyzer.run(dem=eggbox, locations_site=sites, locations_reference=sites.iloc[5:, :])

## Assessment

In [ ]:
figure, axes = phoibe.geography.plot.plot_raster(da=eggbox)
rix_results.steep_segments.plot(color="r", ax=axes, linewidth=0.7)
sites.plot(color="b", ax=axes)
for _k, row in sites.iterrows():
    axes.annotate(xy=(row.geometry.x, row.geometry.y), text=row["name"], fontsize=17)

In [ ]:
def plot_polar(rix_directional, label=None, ax=None, figure=None):

    angles_radians = np.deg2rad(np.append(rix_directional.index, rix_directional.index[0]))
    values = rix_directional.values
    values = np.append(values, values[0])

    if ax is None:
        figure, ax = plt.subplots(subplot_kw={"projection": "polar"})

    ax.plot(angles_radians, values, label=label, linewidth=0.7)
    ax.set_theta_zero_location("N")
    ax.set_theta_direction(-1)
    ax.set_ylabel("RIX")

    return figure, ax

In [ ]:
rix_results.steep_segments.loc[rix_results.steep_segments.theta == 90.0]  # .geometry[331].xy
figure, ax = plt.subplots(subplot_kw={"projection": "polar"})
for key, row in rix_results.roses.iterrows():
    plot_polar(row, label=str(key), ax=ax, figure=figure)

ax.legend()
ax.set_title("RIX roses")

In [ ]:
display(rix_results.summary)

In [ ]:
rix_results.transferability

In [ ]:
rix_results.meta

## Helper functions

### Plotting

In [ ]:
def plot_profile_and_steep_segments(ray_profile, slope_critical):
    """Plot the elevation profile and its slopes along a ray, and marks steep parts."""
    r_m = ray_profile.r_m[:-1]
    z = ray_profile.z[:-1]
    dz = analyse.slopes(ray_profile)
    steep_mask = analyse.steep_mask(ray_profile, slope_critical=slope_critical)

    figure, axes = plt.subplots(1, 2, squeeze=True, figsize=(17, 7))
    axes[0].plot(r_m, z)
    axes[0].plot(r_m[steep_mask], z[steep_mask], c="r")
    axes[0].set_title(f"Profile along ray for {slope_critical=}")

    axes[1].plot(r_m, dz)
    axes[1].hlines(y=slope_critical, xmin=r_m[0], xmax=r_m[-1], color="r")
    axes[1].hlines(y=-slope_critical, xmin=r_m[0], xmax=r_m[-1], color="r")
    axes[1].set_title(f"Slope along ray for {slope_critical=}")

    return figure, axes

### Computation and evaluation of intermediate and final results

In [ ]:
def evaluate_rix_for_ray(ray_profile: RayProfile, slope_critical: float):
    """Evaluate RIX and related numbers."""
    n_steep_segments = analyse.steep_mask(ray_profile, slope_critical).sum()
    n_segments = analyse.slopes(ray_profile).size
    rix = RayResult(profile=ray_profile, slope_critical=slope_critical).ruggedness
    message = (
        f"Number of steep segments {n_steep_segments} vs total number of segments {n_segments} vs. RIX {rix:2.4f}."
    )
    return message

## Reprojections

A more detailed inspection of site 3:
- Ray facing south.
- Interpolation method:
  - nearest: Spiky as profile is step function. Resolution needs to be set properly.
  - linear: Unexpected changes in steepness of the profile leads to jumps in the slope.

In [ ]:
site = sites.loc[3]

LOCATION = ergaleiothiki.perdix.LocationCCS(easting=site.geometry.x, northing=site.geometry.y, zone=33)
THETA = 180
INTERPOLATION_METHOD = "linear"


ray = RayGeometry.from_compass_regular(location=LOCATION, theta=THETA, R_km=config["R_km"], dr_km=config["dr_km"])
display("Coordinate arrays:")
display(ray.xs, ray.ys)
elevation_sampler = RegularGridXYSampler(da=eggbox, method=INTERPOLATION_METHOD)
ray_profile = RayProfile.create_regular(ray=ray, sampler=elevation_sampler, nan_policy=NaNPolicy.ERROR)
display("RayProfile (regular)")
display(ray_profile.z)


RayResult(profile=ray_profile, slope_critical=0.013).describe()
ys = RayResult(profile=ray_profile, slope_critical=0.013).steep_segments_geodataframe(crs=sites.crs).geometry[0].xy[1]
print(np.array(ys[1:]) - np.array(ys[:-1]), min(ys), max(ys))

ys = RayResult(profile=ray_profile, slope_critical=0.013).steep_segments_geodataframe(crs=sites.crs).geometry[1].xy[1]
print(np.array(ys[1:]) - np.array(ys[:-1]), min(ys), max(ys))

_ = plot_profile_and_steep_segments(ray_profile, 0.013)